# Lyrical Emotion Analyzer — Training on Colab

**Before running:** Make sure you have selected a T4 GPU runtime.
Runtime → Change runtime type → T4 GPU

**Files to upload to Google Drive first:**
```
processed/train.pt
processed/val.pt
processed/test.pt
processed/scaler.pkl
model.py
train.py
preprocess.py
```
Upload them inside a folder called `LyricalEmotionAnalyzer` in your Google Drive.

In [ ]:
# ── Step 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Step 2: Set project folder path ──────────────────────────────────────────
import os

PROJECT_DIR = '/content/drive/MyDrive/LyricalEmotionAnalyzer'
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
print(f'Files found: {os.listdir()}')

In [ ]:
# ── Step 3: Install dependencies ─────────────────────────────────────────────
!pip install -q transformers==4.40.0 torch torchvision torchaudio
print('Dependencies installed.')

In [ ]:
# ── Step 4: Verify GPU ────────────────────────────────────────────────────────
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

if not torch.cuda.is_available():
    print('\n⚠️  No GPU detected! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Step 5: Verify processed files exist ─────────────────────────────────────
required = [
    'processed/train.pt',
    'processed/val.pt',
    'processed/test.pt',
    'processed/scaler.pkl',
    'model.py',
    'train.py',
    'preprocess.py',
]
all_ok = True
for f in required:
    exists = os.path.exists(f)
    status = '✓' if exists else '✗ MISSING'
    print(f'{status}  {f}')
    if not exists:
        all_ok = False

if all_ok:
    print('\nAll files present. Ready to train.')
else:
    print('\n⚠️  Upload missing files to your Google Drive folder before continuing.')

In [ ]:
# ── Step 6: Start training ────────────────────────────────────────────────────
# Recommended: top 6 RoBERTa layers, early stopping patience 7
# Expected: ~5-10 min/epoch on T4 → 1-3 hours total

!python3 train.py --unfreeze-layers 6 --epochs 30 --batch-size 16

In [ ]:
# ── Step 7: Plot training curves after training ───────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv('checkpoints/training_log.csv')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training Results', fontsize=14, fontweight='bold')

# Loss
axes[0].plot(log['epoch'], log['train_total'], label='Train')
axes[0].plot(log['epoch'], log['val_total'], label='Val')
axes[0].set_title('Total Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Decade accuracy
axes[1].plot(log['epoch'], log['val_decade_acc'] * 100, color='green')
axes[1].axhline(y=14.3, color='red', linestyle='--', label='Chance (14.3%)')
axes[1].set_title('Decade Classification Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Sigma values
if 'sigma_valence' in log.columns:
    for col, label in [('sigma_valence', 'Valence'), ('sigma_energy', 'Energy'),
                       ('sigma_danceability', 'Dance'), ('sigma_tempo', 'Tempo'),
                       ('sigma_decade', 'Decade')]:
        axes[2].plot(log['epoch'], log[col], label=label)
    axes[2].set_title('Task Uncertainty (σ)')
    axes[2].set_xlabel('Epoch')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('checkpoints/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → checkpoints/training_curves.png')

In [ ]:
# ── Step 8: Print final results ───────────────────────────────────────────────
log = pd.read_csv('checkpoints/training_log.csv')
best = log.loc[log['val_total'].idxmin()]

print('=' * 50)
print('  TRAINING COMPLETE')
print('=' * 50)
print(f'Best epoch:        {int(best["epoch"])}')
print(f'Best val loss:     {best["val_total"]:.4f}')
print(f'Best decade acc:   {best["val_decade_acc"]:.2%}')
print(f'Val valence MSE:   {best["val_valence"]:.4f}')
print()
print('Checkpoint saved → checkpoints/best_model.pt')
print('Next step: run evaluate_plots.py')